In [ ]:
# Cell 1: Imports & config

import random
import time
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import make_pipeline

random.seed(42)


In [ ]:
# Cell 2: Define SAFE query pool

safe_pool = [
"db.users.find({\"username\": \"user1\"})",
"db.users.find({\"username\": \"user2\"})",
"db.users.find({\"age\": {\"$gt\": 21}})",
"db.users.find({\"email\": \"test1@example.com\"})",
"db.users.find({\"email\": \"test2@example.com\"})",
"db.users.find({\"active\": true})",
"db.users.find({\"active\": false})",
"db.users.find({\"role\": \"member\"})",
"db.users.find({\"role\": \"customer\"})",
"db.users.find({\"created\": {\"$gt\": 100}})",

"db.orders.find({\"status\": \"shipped\"})",
"db.orders.find({\"status\": \"processing\"})",
"db.orders.find({\"status\": \"pending\"})",
"db.orders.find({\"total\": {\"$lt\": 50}})",
"db.orders.find({\"total\": {\"$gt\": 100}})",
"db.orders.find({\"customerId\": 456})",
"db.orders.find({\"customerId\": 789})",
"db.orders.find({\"quantity\": {\"$gte\": 5}})",
"db.orders.find({\"quantity\": {\"$lte\": 10}})",
"db.orders.find({\"shipping\": \"ground\"})",

"db.products.find({\"category\": \"electronics\"})",
"db.products.find({\"category\": \"toys\"})",
"db.products.find({\"category\": \"books\"})",
"db.products.find({\"price\": {\"$gte\": 20}})",
"db.products.find({\"price\": {\"$lte\": 40}})",
"db.products.find({\"inStock\": true})",
"db.products.find({\"inStock\": false})",
"db.products.find({\"brand\": \"brandA\"})",
"db.products.find({\"brand\": \"brandB\"})",
"db.products.find({\"rating\": {\"$gt\": 3}})",

"db.logs.insertOne({\"event\": \"login\", \"user\": \"user1\"})",
"db.logs.insertOne({\"event\": \"logout\", \"user\": \"user2\"})",
"db.logs.insertOne({\"event\": \"login\", \"user\": \"user3\"})",
"db.logs.insertOne({\"event\": \"update\", \"user\": \"user4\"})",
"db.logs.insertOne({\"event\": \"download\", \"user\": \"user5\"})",
"db.logs.insertOne({\"event\": \"create\", \"user\": \"user6\"})",
"db.logs.insertOne({\"event\": \"upload\", \"user\": \"user7\"})",
"db.logs.insertOne({\"event\": \"payment\", \"user\": \"user8\"})",
"db.logs.insertOne({\"event\": \"report\", \"user\": \"user9\"})",
"db.logs.insertOne({\"event\": \"support\", \"user\": \"user10\"})",

"db.sessions.deleteMany({\"expired\": true})",
"db.sessions.deleteMany({\"expired\": false})",
"db.sessions.deleteMany({\"userId\": 12})",
"db.sessions.deleteMany({\"userId\": 24})",
"db.sessions.deleteMany({\"userId\": 36})",
"db.sessions.deleteMany({\"userId\": 48})",
"db.sessions.deleteMany({\"failed\": true})",
"db.sessions.deleteMany({\"failed\": false})",
"db.sessions.deleteMany({\"device\": \"mobile\"})",
"db.sessions.deleteMany({\"device\": \"desktop\"})",

"db.inventory.find({\"sku\": \"SKU10\", \"quantity\": {\"$gte\": 10}})",
"db.inventory.find({\"sku\": \"SKU11\", \"quantity\": {\"$gte\": 11}})",
"db.inventory.find({\"sku\": \"SKU12\", \"quantity\": {\"$gte\": 12}})",
"db.inventory.find({\"sku\": \"SKU13\", \"quantity\": {\"$gte\": 13}})",
"db.inventory.find({\"sku\": \"SKU14\", \"quantity\": {\"$gte\": 14}})",
"db.inventory.find({\"sku\": \"SKU15\", \"quantity\": {\"$gte\": 15}})",
"db.inventory.find({\"sku\": \"SKU16\", \"quantity\": {\"$gte\": 16}})",
"db.inventory.find({\"sku\": \"SKU17\", \"quantity\": {\"$gte\": 17}})",
"db.inventory.find({\"sku\": \"SKU18\", \"quantity\": {\"$gte\": 18}})",
"db.inventory.find({\"sku\": \"SKU19\", \"quantity\": {\"$gte\": 19}})",

"db.payments.find({\"method\": \"credit_card\"})",
"db.payments.find({\"method\": \"cash\"})",
"db.payments.find({\"method\": \"debit\"})",
"db.payments.find({\"amount\": {\"$gt\": 25}})",
"db.payments.find({\"amount\": {\"$lt\": 60}})",
"db.payments.find({\"currency\": \"USD\"})",
"db.payments.find({\"currency\": \"EUR\"})",
"db.payments.find({\"exchange\": \"domestic\"})",
"db.payments.find({\"gateway\": \"stripe\"})",
"db.payments.find({\"gateway\": \"paypal\"})",

"db.users.updateOne({\"username\": \"userA\"}, {\"$set\": {\"email\": \"userA@example.com\"}})",
"db.users.updateOne({\"username\": \"userB\"}, {\"$set\": {\"email\": \"userB@example.com\"}})",
"db.users.updateOne({\"username\": \"userC\"}, {\"$set\": {\"email\": \"userC@example.com\"}})",
"db.users.updateOne({\"username\": \"userD\"}, {\"$set\": {\"email\": \"userD@example.com\"}})",
"db.users.updateOne({\"username\": \"userE\"}, {\"$set\": {\"email\": \"userE@example.com\"}})",
"db.users.updateOne({\"username\": \"userF\"}, {\"$set\": {\"email\": \"userF@example.com\"}})",
"db.users.updateOne({\"username\": \"userG\"}, {\"$set\": {\"email\": \"userG@example.com\"}})",
"db.users.updateOne({\"username\": \"userH\"}, {\"$set\": {\"email\": \"userH@example.com\"}})",
"db.users.updateOne({\"username\": \"userI\"}, {\"$set\": {\"email\": \"userI@example.com\"}})",
"db.users.updateOne({\"username\": \"userJ\"}, {\"$set\": {\"email\": \"userJ@example.com\"}})",

"db.customers.find({\"vip\": true})",
"db.customers.find({\"vip\": false})",
"db.customers.find({\"loyalty\": \"silver\"})",
"db.customers.find({\"loyalty\": \"gold\"})",
"db.customers.find({\"loyalty\": \"bronze\"})",
"db.customers.find({\"lastPurchase\": {\"$gt\": 20}})",
"db.customers.find({\"lastPurchase\": {\"$lt\": 50}})",
"db.customers.find({\"country\": \"US\"})",
"db.customers.find({\"country\": \"CA\"})",
"db.customers.find({\"newsletter\": true})",

"db.audit.find({\"action\": \"create\"})",
"db.audit.find({\"action\": \"update\"})",
"db.audit.find({\"action\": \"delete\"})",
"db.audit.find({\"user\": \"admin\"})",
"db.audit.find({\"user\": \"system\"})",
"db.audit.find({\"event\": \"login\"})",
"db.audit.find({\"event\": \"logout\"})",
"db.audit.find({\"event\": \"password_reset\"})",
"db.audit.find({\"event\": \"checkout\"})",
"db.audit.find({\"event\": \"download\"})"

]


print("Safe pool size:", len(safe_pool))


Safe pool size: 100


In [ ]:
# Cell 3: Define MALICIOUS query pool

malicious_pool = [
"db.users.find({\"$where\": \"this.password == '111'\"})",
"db.users.find({\"username\": \"admin\" || \"1\"==\"1\"})",
"db.users.find({\"$or\": [ {}, {\"role\": \"admin\"} ] })",
"db.accounts.find({\"$where\": \"this.balance < 0 || 1==1\"})",
"db.collection.find({\"$where\": \"while(true){}\"})",
"db.login.find({\"user\": \"x\", \"pass\": \"x\"}) || db.admin.drop()",
"db.users.find({\"name\": {\"$ne\": null}, \"$where\": \"this.isAdmin == true\" })",
"db.customers.find({\"$where\": \"this.ssn != null\"})",
"db.products.find({\"$where\": \"this.price == 0 || 1==1\"})",
"db.audit.find({\"$where\": \"this.action == 'delete' && this.user == 'admin'\"})",

"db.users.find({\"$where\": \"this.password == '222'\"})",
"db.users.find({\"username\": \"root\" || \"1\"==\"1\"})",
"db.users.find({\"$or\": [ {\"role\": \"superuser\"}, {} ] })",
"db.accounts.find({\"$where\": \"this.balance <= 0 || 1==1\"})",
"db.collection.find({\"$where\": \"for(;;){}\"})",
"db.login.find({\"user\": \"y\", \"pass\": \"x\"}) || db.admin.drop()",
"db.users.find({\"name\": {\"$ne\": null}, \"$where\": \"this.isAdmin == true || 1==1\" })",
"db.customers.find({\"$where\": \"this.cardNumber != null\"})",
"db.products.find({\"$where\": \"this.stock == 0 || 1==1\"})",
"db.audit.find({\"$where\": \"this.action == 'update' && this.user == 'admin'\"})",

"db.users.find({\"$where\": \"this.password == '333'\"})",
"db.users.find({\"username\": \"test\" || \"1\"==\"1\"})",
"db.users.find({\"$or\": [ {\"role\": \"owner\"}, {} ] })",
"db.accounts.find({\"$where\": \"this.balance < 100 || 1==1\"})",
"db.collection.find({\"$where\": \"while(1){}\"})",
"db.login.find({\"user\": \"z\", \"pass\": \"x\"}) || db.admin.drop()",
"db.users.find({\"name\": {\"$ne\": null}, \"$where\": \"this.isAdmin == false || 1==1\" })",
"db.customers.find({\"$where\": \"this.ssn != ''\"})",
"db.products.find({\"$where\": \"this.price > 0 || 1==1\"})",
"db.audit.find({\"$where\": \"this.action == 'insert' && this.user == 'admin'\"})",

"db.users.find({\"$where\": \"this.password == '444'\"})",
"db.users.find({\"username\": \"admin\" || \"true\"==\"true\"})",
"db.users.find({\"$or\": [ {\"role\": \"manager\"}, {} ] })",
"db.accounts.find({\"$where\": \"this.balance != 0 || 1==1\"})",
"db.collection.find({\"$where\": \"do{}while(true)\"})",
"db.login.find({\"user\": \"t\", \"pass\": \"x\"}) || db.admin.drop()",
"db.users.find({\"name\": {\"$ne\": null}, \"$where\": \"this.isAdmin != false\" })",
"db.customers.find({\"$where\": \"this.email != null\"})",
"db.products.find({\"$where\": \"this.price == null || 1==1\"})",
"db.audit.find({\"$where\": \"this.action == 'read' && this.user == 'admin'\"})",

"db.users.find({\"$where\": \"this.password == '555'\"})",
"db.users.find({\"username\": \"root\" || 1==1})",
"db.users.find({\"$or\": [ {\"role\": \"dev\"}, {} ] })",
"db.accounts.find({\"$where\": \"this.balance >= 0 || 1==1\"})",
"db.collection.find({\"$where\": \"if(true){}\"})",
"db.login.find({\"user\": \"u\", \"pass\": \"x\"}) || db.admin.drop()",
"db.users.find({\"name\": {\"$ne\": null}, \"$where\": \"this.isAdmin == true && 1==1\" })",
"db.customers.find({\"$where\": \"this.phone != null\"})",
"db.products.find({\"$where\": \"this.discount > 0 || 1==1\"})",
"db.audit.find({\"$where\": \"this.action == 'drop' && this.user == 'admin'\"})",

"db.users.find({\"$where\": \"this.password == '666'\"})",
"db.users.find({\"username\": \"sys\" || \"1\"==\"1\"})",
"db.users.find({\"$or\": [ {\"role\": \"support\"}, {} ] })",
"db.accounts.find({\"$where\": \"this.balance > 0 || 1==1\"})",
"db.collection.find({\"$where\": \"for(x in [1,2,3]){}\"})",
"db.login.find({\"user\": \"q\", \"pass\": \"x\"}) || db.admin.drop()",
"db.users.find({\"name\": {\"$ne\": null}, \"$where\": \"this.isAdmin == true || this.user == 'admin'\" })",
"db.customers.find({\"$where\": \"this.address != null\"})",
"db.products.find({\"$where\": \"this.category == 'secret' || 1==1\"})",
"db.audit.find({\"$where\": \"this.action == 'config' && this.user == 'admin'\"})",

"db.users.find({\"$where\": \"this.password == '777'\"})",
"db.users.find({\"username\": \"guest\" || \"1\"==\"1\"})",
"db.users.find({\"$or\": [ {\"role\": \"guest\"}, {} ] })",
"db.accounts.find({\"$where\": \"this.balance < 1000 || 1==1\"})",
"db.collection.find({\"$where\": \"switch(1){case 1:break;}\"})",
"db.login.find({\"user\": \"p\", \"pass\": \"x\"}) || db.admin.drop()",
"db.users.find({\"name\": {\"$ne\": null}, \"$where\": \"this.isAdmin == true || this.role == 'admin'\" })",
"db.customers.find({\"$where\": \"this.zip != null\"})",
"db.products.find({\"$where\": \"this.sku == 'secret' || 1==1\"})",
"db.audit.find({\"$where\": \"this.action == 'backup' && this.user == 'admin'\"})",

"db.users.find({\"$where\": \"this.password == '888'\"})",
"db.users.find({\"username\": \"root\" || \"x\"==\"x\"})",
"db.users.find({\"$or\": [ {\"role\": \"hacker\"}, {} ] })",
"db.accounts.find({\"$where\": \"this.balance <= 100 || 1==1\"})",
"db.collection.find({\"$where\": \"try{}catch(e){}\"})",
"db.login.find({\"user\": \"o\", \"pass\": \"x\"}) || db.admin.drop()",
"db.users.find({\"name\": {\"$ne\": null}, \"$where\": \"this.isAdmin == true || this.flag == 'root'\" })",
"db.customers.find({\"$where\": \"this.token != null\"})",
"db.products.find({\"$where\": \"this.internal == true || 1==1\"})",
"db.audit.find({\"$where\": \"this.action == 'override' && this.user == 'admin'\"})",

"db.users.find({\"$where\": \"this.password == '999'\"})",
"db.users.find({\"username\": \"admin\" || \"0\"==\"0\"})",
"db.users.find({\"$or\": [ {\"role\": \"operator\"}, {} ] })",
"db.accounts.find({\"$where\": \"this.balance != null || 1==1\"})",
"db.collection.find({\"$where\": \"if(1){while(true){}}\"})",
"db.login.find({\"user\": \"n\", \"pass\": \"x\"}) || db.admin.drop()",
"db.users.find({\"name\": {\"$ne\": null}, \"$where\": \"this.isAdmin == true || this.session == 'elevated'\" })",
"db.customers.find({\"$where\": \"this.secret != null\"})",
"db.products.find({\"$where\": \"this.hidden == true || 1==1\"})",
"db.audit.find({\"$where\": \"this.action == 'shutdown' && this.user == 'admin'\"})"


]

print("Malicious pool size:", len(malicious_pool))


Malicious pool size: 90


In [ ]:
# Cell 4: Sample 90 safe + 60 malicious, build df

# sanity check
assert len(safe_pool) >= 90, "Need at least 90 safe queries"
assert len(malicious_pool) >= 60, "Need at least 60 malicious queries"

safe_sample = random.sample(safe_pool, 90)
malicious_sample = random.sample(malicious_pool, 60)

queries = safe_sample + malicious_sample
labels = ["safe"] * 90 + ["malicious"] * 60

df = pd.DataFrame({"query_text": queries, "label": labels})

# Shuffle dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Dataset shape:", df.shape)
print(df["label"].value_counts())
df.head()


Dataset shape: (150, 2)
label
safe         90
malicious    60
Name: count, dtype: int64


,query_text,label
0,"db.customers.find({""loyalty"": ""gold""})",safe
1,"db.products.find({""rating"": {""$gt"": 3}})",safe
2,"db.users.find({""$or"": [ {""role"": ""manager""}, {...",malicious
3,"db.products.find({""category"": ""toys""})",safe
4,"db.audit.find({""event"": ""logout""})",safe


In [ ]:
# Cell 5: 60/40 train–test split + TF-IDF

X = df["query_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.4,      # 60% train, 40% test
    stratify=y,
    random_state=42
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

print("Train TF-IDF shape:", X_train_vec.shape)
print("Test TF-IDF shape:", X_test_vec.shape)


Train size: 90
Test size: 60
Train TF-IDF shape: (90, 147)
Test TF-IDF shape: (60, 147)


In [ ]:
# Cell 6: Train models and print reports + confusion matrices

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "Linear SVM": LinearSVC()
}

trained_models = {}

for name, clf in models.items():
    clf.fit(X_train_vec, y_train)
    y_pred = clf.predict(X_test_vec)
    trained_models[name] = clf

    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred))
    print("Confusion matrix (rows=true, cols=pred):")
    print(confusion_matrix(y_test, y_pred))



=== Logistic Regression ===
              precision    recall  f1-score   support

   malicious       0.91      0.88      0.89        24
        safe       0.92      0.94      0.93        36

    accuracy                           0.92        60
   macro avg       0.92      0.91      0.91        60
weighted avg       0.92      0.92      0.92        60

Confusion matrix (rows=true, cols=pred):
[[21  3]
 [ 2 34]]

=== Random Forest ===
              precision    recall  f1-score   support

   malicious       0.92      0.96      0.94        24
        safe       0.97      0.94      0.96        36

    accuracy                           0.95        60
   macro avg       0.95      0.95      0.95        60
weighted avg       0.95      0.95      0.95        60

Confusion matrix (rows=true, cols=pred):
[[23  1]
 [ 2 34]]

=== Linear SVM ===
              precision    recall  f1-score   support

   malicious       0.92      1.00      0.96        24
        safe       1.00      0.94      0.97  

In [ ]:
# Cell 7: 5-fold cross-validation for robustness

print("\n=== 5-fold cross-validation accuracy (TF-IDF + model) ===")

for name, clf in models.items():
    pipe = make_pipeline(TfidfVectorizer(), clf)
    scores = cross_val_score(pipe, X, y, cv=5, scoring="accuracy")
    print(f"{name}: mean={scores.mean():.3f}, std={scores.std():.3f}, scores={scores}")



=== 5-fold cross-validation accuracy (TF-IDF + model) ===
Logistic Regression: mean=0.953, std=0.040, scores=[1.         0.93333333 0.9        1.         0.93333333]
Random Forest: mean=0.960, std=0.039, scores=[0.9        1.         0.93333333 0.96666667 1.        ]
Linear SVM: mean=0.973, std=0.025, scores=[0.93333333 0.96666667 0.96666667 1.         1.        ]


In [ ]:
# Cell 8: Per-query latency measurement

def measure_latency(model, vectorizer, query_text, n_runs=1000):
    vec = vectorizer.transform([query_text])
    start = time.perf_counter()
    for _ in range(n_runs):
        model.predict(vec)
    end = time.perf_counter()
    return (end - start) / n_runs

test_query = 'db.users.find({ "username": "marco", "active": true })'

print("\n=== Approximate per-query prediction latency (seconds) ===")
for name, clf in trained_models.items():
    lat = measure_latency(clf, vectorizer, test_query, n_runs=1000)
    print(f"{name}: {lat:.6e} sec/query")



=== Approximate per-query prediction latency (seconds) ===
Logistic Regression: 2.392223e-04 sec/query
Random Forest: 2.569782e-02 sec/query
Linear SVM: 2.156686e-04 sec/query


In [ ]:
# Cell 9: Unified predict_query() helper for demo

def predict_query(query_text: str):
    """
    Classify a single MongoDB query string as 'safe' or 'malicious'
    using all three trained models.
    """
    vec = vectorizer.transform([query_text])
    return {
        "LogisticRegression": trained_models["Logistic Regression"].predict(vec)[0],
        "RandomForest":       trained_models["Random Forest"].predict(vec)[0],
        "LinearSVM":          trained_models["Linear SVM"].predict(vec)[0]
    }

# Example predictions
safe_example = 'db.users.find({ "username": "test_user" })'
mal_example  = 'db.users.find({ "$where": "this.isAdmin == true || 1==1" })'

print("Safe example:")
print(safe_example)
print(predict_query(safe_example))

print("\nMalicious example:")
print(mal_example)
print(predict_query(mal_example))


Safe example:
db.users.find({ "username": "test_user" })
{'LogisticRegression': 'safe', 'RandomForest': 'malicious', 'LinearSVM': 'malicious'}

Malicious example:
db.users.find({ "$where": "this.isAdmin == true || 1==1" })
{'LogisticRegression': 'malicious', 'RandomForest': 'malicious', 'LinearSVM': 'malicious'}


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

query_box = widgets.Textarea(
    value="db.users.find({\"username\": \"user1\"})",
    placeholder="Paste a MongoDB query here...",
    description="Query:",
    layout=widgets.Layout(width="100%", height="120px")
)

preset = widgets.Dropdown(
    options={
        "Safe CRUD": "db.users.find({\"username\": \"user1\"})",
        "Tautology ($where)": "db.users.find({\"$where\": \"1==1\"})",
        "OR injection": "db.users.find({\"$or\": [ {}, {\"role\": \"admin\"} ] })",
        "Infinite loop": "db.collection.find({\"$where\": \"while(true){}\"})",
    },
    description="Preset:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="60%")
)

topk_slider = widgets.IntSlider(
    value=8, min=3, max=20, step=1,
    description="Explain top terms:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="60%")
)

run_btn = widgets.Button(description="Run Prediction", button_style="primary")
showlog_btn = widgets.Button(description="Show Log")
clearlog_btn = widgets.Button(description="Clear Log", button_style="warning")

out = widgets.Output()


def _fmt(x):
    if isinstance(x, (int, float)) and not pd.isna(x):
        if 0 <= x <= 1:
            return f"{x:.2f}"
        return f"{x:.3f}"
    return "" if x is None else x


def on_preset_change(change):
    if change["name"] == "value":
        query_box.value = change["new"]

preset.observe(on_preset_change)


def on_run(_):
    with out:
        clear_output()
        q = query_box.value.strip()
        if not q:
            print("Please enter a query.")
            return

        result = predict_query_majority(q)
        conf = compute_confidences(q)
        log_result(q, result, conf)

        # Table view
        rows = [
            {"Model": "Logistic Regression", "Prediction": result["LogisticRegression"], "Confidence/Score": _fmt(conf["Logistic Regression"]["confidence"])},
            {"Model": "Random Forest",       "Prediction": result["RandomForest"],       "Confidence/Score": _fmt(conf["Random Forest"]["confidence"])},
            {"Model": "Linear SVM",          "Prediction": result["LinearSVM"],          "Confidence/Score": _fmt(conf["Linear SVM"]["score"])},
            {"Model": "Majority Vote",       "Prediction": result["MajorityVote"],       "Confidence/Score": ""},
        ]
        df_view = pd.DataFrame(rows)

        print("Query:")
        print(q)
        print("\nPredictions:")
        display(df_view)

        # Explanation
        top_terms = explain_top_terms(q, top_k=int(topk_slider.value))
        if top_terms:
            print("\nTop contributing terms (Logistic Regression):")
            for term, score in top_terms:
                sign = "+" if score >= 0 else "-"
                print(f"  {sign} {term:<25} {abs(score):.4f}")
        else:
            print("\nTop contributing terms: (not available).")


def on_show_log(_):
    with out:
        clear_output()
        df_log = get_log_df()
        if df_log.empty:
            print("Log is empty.")
        else:
            display(df_log.iloc[::-1].reset_index(drop=True))


def on_clear_log(_):
    with out:
        clear_output()
        PREDICTION_LOG.clear()
        print("Log cleared.")


run_btn.on_click(on_run)
showlog_btn.on_click(on_show_log)
clearlog_btn.on_click(on_clear_log)

display(widgets.HBox([preset, run_btn, showlog_btn, clearlog_btn]), topk_slider, query_box, out)

IntSlider(value=8, description='Explain top terms:', layout=Layout(width='60%'), max=20, min=3, style=SliderSt…

Textarea(value='db.users.find({"username": "user1"})', description='Query:', layout=Layout(height='120px', wid…

Output()